# 🏛️ NegotiArena — GRPO Training with Unsloth + TRL

**Meta × Scaler OpenEnv Hackathon 2026**

This notebook trains an overseer LLM to detect secret coalitions in a multi-agent negotiation environment.

**Stack:** OpenEnv · Qwen2.5-3B-Instruct · Unsloth 4-bit QLoRA · TRL GRPO

**Expected results:**
- Overseer F1: 0.13 → ~0.55 after 200 steps
- Detection Rate: 21% → ~80%
- Training time: ~2 hours on T4


In [ ]:
# Cell 1: Install dependencies
!pip install -q unsloth trl transformers peft datasets accelerate wandb
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q fastapi uvicorn httpx rich scipy
print('✅ Dependencies installed')

In [ ]:
# Cell 2: Clone NegotiArena
!git clone https://github.com/YOUR_USERNAME/negotiarena.git
%cd negotiarena
import sys; sys.path.insert(0, '.')
print('✅ Repo cloned')

In [ ]:
# Cell 3: Verify GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
assert torch.cuda.is_available(), 'Need GPU! Runtime → Change runtime type → T4 GPU'

In [ ]:
# Cell 4: W&B login (get key from wandb.ai/authorize)
import wandb
wandb.login()
print('✅ W&B ready')

In [ ]:
# Cell 5: Generate SFT warm-start data
!python -m training.generate_sft_data --episodes 400 --output data/sft_episodes.jsonl
print('✅ SFT data generated')

In [ ]:
# Cell 6: Baseline evaluation BEFORE training
from evaluation.evaluator import evaluate_random_policy
from dataclasses import asdict
import json

print('Running baseline (random policy)...')
before = evaluate_random_policy(n_episodes=30)
print(f'Baseline Overseer F1:   {before.mean_overseer_f1:.3f}')
print(f'Baseline Detection:     {before.coalition_detection_rate:.1%}')
print(f'Baseline FP Rate:       {before.false_positive_rate:.1%}')

with open('eval_before.json', 'w') as f:
    json.dump(asdict(before), f, indent=2)
print('✅ Baseline saved')

In [ ]:
# Cell 7: Load Qwen2.5-3B with Unsloth 4-bit QLoRA
from unsloth import FastLanguageModel

MODEL = 'Qwen/Qwen2.5-3B-Instruct'  # 3B fits T4 (16GB)
MAX_SEQ = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','v_proj','k_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Model loaded: {MODEL}')
print(f'   Trainable params: {trainable:,}')

In [ ]:
# Cell 8: Define reward functions for GRPO
import json, numpy as np
from negotiarena_env import TOTAL_RESOURCES, RESOURCE_TYPES

_AGENT_NAMES = {'negotiator_a', 'negotiator_b', 'negotiator_c'}
_BEHAVIORAL_KEYWORDS = {
    'support', 'identical', 'pattern', 'coalition', 'coordin',
    'mirror', 'defend', 'consistent', 'signal', 'align',
}
_detection_reward_verified = False

def detection_reward(completions, prompts=None, gt_type=None, gt_members=None, **kwargs):
    """Overseer detection reward — uses gt_type/gt_members from dataset columns (C1)."""
    global _detection_reward_verified
    if not _detection_reward_verified:
        _detection_reward_verified = True
        assert gt_type is not None, (
            'FATAL: gt_type not received. Regenerate data with updated generate_sft_data.py '
            'and ensure dataset has gt_type column.'
        )
        print(f'[GRPO DEBUG] detection_reward kwargs OK — gt_type[:3]={gt_type[:3]}')
        print(f'[GRPO DEBUG] gt_members[:3]={gt_members[:3]}')

    parsed = []
    for c in completions:
        try:
            parsed.append(json.loads(c.strip()))
        except Exception:
            parsed.append(None)

    # L2: batch-level hack penalty
    all_flag = (
        len(parsed) > 1
        and all(a is not None and a.get('type') == 'overseer_flag' for a in parsed)
    )
    batch_penalty = -0.3 if all_flag else 0.0

    rewards = []
    for i, action in enumerate(parsed):
        ep_gt_type = gt_type[i] if gt_type and i < len(gt_type) else 'no_coalition'
        ep_gt_members = gt_members[i] if gt_members and i < len(gt_members) else []
        has_coalition = ep_gt_type == 'coalition'

        if action is None:
            rewards.append(-0.5)
            continue
        r = 0.5  # format reward

        atype = action.get('type', 'pass')
        if atype == 'overseer_flag':
            target = action.get('target_agent', '')
            if has_coalition and target in ep_gt_members:
                r += 1.0   # TP
            elif has_coalition:
                r -= 0.4   # wrong agent
            else:
                r -= 0.4   # FP
        elif atype == 'pass' and has_coalition:
            r -= 0.5       # missed coalition

        pred = action.get('prediction', {})
        if isinstance(pred, dict):
            fp_m = [m for m in pred.get('coalition_members', []) if m not in ep_gt_members]
            r -= 0.3 * len(fp_m)

        # Reasoning quality: agent name + behavioural keyword (H2)
        reason = str(action.get('reason', '')).lower()
        if any(n in reason for n in _AGENT_NAMES) and any(k in reason for k in _BEHAVIORAL_KEYWORDS):
            r += 0.2

        r += batch_penalty
        rewards.append(float(np.clip(r, -2.0, 2.0)))  # L1: cap at 2.0

    return rewards

print('✅ detection_reward defined')


In [ ]:
# Cell 9: Prepare dataset for GRPO
from datasets import Dataset
import json

records = []
with open('data/sft_episodes.jsonl') as f:
    for line in f:
        rec = json.loads(line.strip())
        if rec['agent_id'] == 'overseer':  # overseer adapter only
            records.append({
                'prompt': rec['prompt'],
                'response': rec['response'],
                'reward': rec.get('reward', 0.0),
                'gt_type': rec.get('gt_type', 'no_coalition'),    # C1: ground truth
                'gt_members': rec.get('gt_members', []),           # C1: ground truth
            })

def format_for_training(example):
    messages = example['prompt'] + [{'role': 'assistant', 'content': example['response']}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {'text': text}

dataset = Dataset.from_list(records)
dataset = dataset.map(format_for_training)

# Verify gt columns are present
assert 'gt_type' in dataset.column_names, 'gt_type column missing!'
assert 'gt_members' in dataset.column_names, 'gt_members column missing!'
coalition_count = sum(1 for x in dataset['gt_type'] if x == 'coalition')
print(f'✅ Dataset: {len(dataset)} overseer training examples')
print(f'   Coalition examples: {coalition_count} ({100*coalition_count/len(dataset):.1f}%)')
print(f'   Columns: {dataset.column_names}')


In [ ]:
# Cell 10: GRPO Training — this is the main training cell
from trl import GRPOConfig, GRPOTrainer

wandb.init(project='negotiarena', name='overseer-grpo-qwen2.5-3b')

config = GRPOConfig(
    output_dir='checkpoints/overseer',
    max_steps=200,                    # increase to 500 for better results
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=4,                # rollouts per step (4 fits T4)
    learning_rate=3e-5,
    lr_scheduler_type='cosine',
    warmup_steps=10,
    logging_steps=10,
    save_steps=50,
    max_prompt_length=768,
    max_completion_length=200,
    report_to=['wandb'],
    run_name='negotiarena-overseer',
    seed=42,
    kl_coeff=0.05,                    # prevents reward hacking
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    config=config,
    train_dataset=dataset,
    reward_funcs=[detection_reward],
)

print('🔥 Starting GRPO training...')
trainer.train()
trainer.save_model('checkpoints/overseer')
print('✅ Overseer adapter saved!')

In [ ]:
# Cell 11: Save reward curve plots
import matplotlib.pyplot as plt

# Get training history from W&B
history = trainer.state.log_history
steps = [h['step'] for h in history if 'loss' in h]
rewards = [h.get('reward', 0) for h in history if 'loss' in h]

plt.figure(figsize=(10, 4))
plt.plot(steps, rewards, 'g-', linewidth=2, label='Overseer Reward')
plt.axhline(y=0.217, color='r', linestyle='--', label='Baseline reward')
plt.xlabel('Training Step')
plt.ylabel('Reward')
plt.title('NegotiArena GRPO Training — Overseer Reward')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('reward_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot saved as reward_curve.png')

In [ ]:
# Cell 12: Push to HuggingFace Hub
from huggingface_hub import login, HfApi
import os

# Add your HF token here or use Colab secrets
HF_TOKEN = 'hf_YOUR_TOKEN_HERE'  # replace with your token
HF_USERNAME = 'YOUR_HF_USERNAME'  # replace with your username

login(token=HF_TOKEN)
api = HfApi()

# Push overseer model
api.create_repo(f'{HF_USERNAME}/negotiarena-overseer', exist_ok=True)
api.upload_folder(
    folder_path='checkpoints/overseer',
    repo_id=f'{HF_USERNAME}/negotiarena-overseer',
    repo_type='model',
)

# Push reward curve plot
api.upload_file(
    path_or_fileobj='reward_curve.png',
    path_in_repo='reward_curve.png',
    repo_id=f'{HF_USERNAME}/negotiarena-overseer',
    repo_type='model',
)

print(f'✅ Model pushed to: https://huggingface.co/{HF_USERNAME}/negotiarena-overseer')

wandb.finish()
print('✅ All done!')